In [0]:
display(dbutils.fs.ls("/Volumes/tnelections/form20_bronze/raw_pdfs/2026/")

In [0]:
import re

files = dbutils.fs.ls("/Volumes/tnelections/form20_bronze/raw_pdfs/2026/")

# anything not named exactly like AC001.pdf
odd = [f.name for f in files if not re.fullmatch(r"AC\d{3}\.pdf", f.name)]
print("Odd names:", odd)

# any AC number from 1–234 that has no file
nums = {int(f.name[2:5]) for f in files if re.fullmatch(r"AC\d{3}\.pdf", f.name)}
print("Missing ACs:", sorted(set(range(1, 235)) - nums))

In [0]:
from pyspark.sql.functions import col, regexp_extract, current_timestamp

raw_path = "/Volumes/tnelections/form20_bronze/raw_pdfs/"
target_table = "tnelections.form20_bronze.pdf_files"

pdf_files = (
    spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.pdf")
    .load(raw_path)
    .withColumn("year",  regexp_extract(col("path"), r"/(\d{4})/[^/]+\.pdf$", 1).try_cast("int"))
    .withColumn("ac_no", regexp_extract(col("path"), r"(?i)AC(\d+)\.pdf$", 1).try_cast("int"))
    .withColumn("ingested_at", current_timestamp())
    .select("path", "year", "ac_no", "length", "content", "ingested_at")
)

pdf_files.write.mode("overwrite").saveAsTable(target_table)

In [0]:
display(spark.sql(f"""
    SELECT
        COUNT(*)                                                     AS total_rows,
        COUNT(DISTINCT ac_no)                                        AS distinct_ac,
        SUM(CASE WHEN year IS NULL OR ac_no IS NULL THEN 1 ELSE 0 END) AS null_keys
    FROM {target_table}
"""))